# `promote_spec` — observable spec promotion through `00_specs/`

This notebook is the **first trigger** for the spec promotion workflow described in
[`00_specs/01_ideas/build-the-promotion-workflow.md`](../00_specs/01_ideas/build-the-promotion-workflow.md).

Each stage gets two cells:
1. **Preview** — assembles the prompt and prints it so you can inspect what the model will see.
2. **Run** — calls the LLM and writes `00_specs/0N_<stage>/<slug>.md` plus a `.meta.yaml` sidecar.

Stop after any stage, edit the artifact by hand on disk, and re-enter at the next stage. Re-running a stage requires `CONFIRM_OVERWRITE = True`.

## 1. Configure the run

In [ ]:
# Which spec are you promoting?  Bare slug only — do NOT include '.md'.
SLUG = "create-format-markdown-workflow"

# Which stage to start at. Stages already on disk are reused as upstream context.
# Valid values: "02_research", "03_requirements", "04_tasks", "05_prompts", "06_final".
START_STAGE = "03_requirements"

# LM Studio (OpenAI-compatible). Override MODEL with whatever you have loaded.
# Ollama works too — point LMSTUDIO_HOST at http://localhost:11434/v1.
LMSTUDIO_HOST = "http://localhost:1234/v1"
MODEL = "qwen/qwen3.6-27b"  # set to whatever model is loaded in LM Studio

# Safety: existing artifacts are NOT overwritten unless this is True.
CONFIRM_OVERWRITE = False

In [ ]:
print("SLUG =", repr(SLUG))
print("chain keys:", list(read_chain(SLUG).keys()))
print("01_ideas contents:", [p.name for p in (SPECS_ROOT / "01_ideas").iterdir()])


SLUG = 'create-format-markdown-workflow'


NameError: name 'SPECS_ROOT' is not defined

## 2. Resolve paths through `core.paths`

We do not climb `__file__` parents — `WORKSPACES_ROOT` is the canonical anchor.

In [ ]:
from pathlib import Path
from core.paths import WORKSPACES_ROOT

SPECS_ROOT = WORKSPACES_ROOT / "00_specs"
PROMPTS_ROOT = WORKSPACES_ROOT / "12_prompts" / "tasks" / "spec_promotion"

assert SPECS_ROOT.is_dir(), f"missing {SPECS_ROOT}"
assert PROMPTS_ROOT.is_dir(), f"missing {PROMPTS_ROOT}"
assert not SLUG.endswith(".md"), (
    f"SLUG must be a bare slug, no .md extension. Got: {SLUG!r}"
)
print(f"specs:   {SPECS_ROOT}")
print(f"prompts: {PROMPTS_ROOT}")

## 3. Stage definitions

The ordered list of stages. The first entry (`01_ideas`) has no prompt — it is the input.

In [3]:
STAGES = [
    ("01_ideas",        None,                                "idea"),
    ("02_research",     PROMPTS_ROOT / "02_research.md",     "research"),
    ("03_requirements", PROMPTS_ROOT / "03_requirements.md", "requirements"),
    ("04_tasks",        PROMPTS_ROOT / "04_tasks.md",        "tasks"),
    ("05_prompts",      PROMPTS_ROOT / "05_prompts.md",      "prompts"),
    ("06_final",        PROMPTS_ROOT / "06_final.md",        "final"),
]

STAGE_BY_FOLDER = {folder: (prompt, var) for folder, prompt, var in STAGES}

## 4. Helpers

- `read_chain(slug)` — collects every upstream artifact already on disk.
- `render_prompt(template, chain)` — substitutes `{{var}}` placeholders.
- `call_llm(prompt)` — single chat completion against the configured endpoint.
- `write_stage(...)` — writes the Markdown artifact and a `.meta.yaml` sidecar.

In [ ]:
import hashlib
import re
import datetime as _dt
import yaml
from openai import OpenAI

_client = OpenAI(base_url=LMSTUDIO_HOST, api_key="not-needed")


def read_chain(slug):
    chain = {}
    for folder, _prompt, var in STAGES:
        path = SPECS_ROOT / folder / f"{slug}.md"
        if path.exists():
            chain[var] = path.read_text()
    return chain


def render_prompt(template_path, chain):
    text = template_path.read_text()
    for key, value in chain.items():
        text = text.replace("{{" + key + "}}", value or "")
    leftover = re.findall(r"\{\{[a-zA-Z_][a-zA-Z0-9_]*\}\}", text)
    if leftover:
        raise ValueError(
            f"Unsubstituted placeholders {leftover}; chain had keys {list(chain.keys())}. "
            f"Likely cause: SLUG points at a spec without the required upstream artifact "
            f"in 00_specs/."
        )
    return text


# Generous default; reasoning models (Qwen3 thinking, DeepSeek R1, etc.) can
# consume thousands of tokens before producing visible content. If LM Studio's
# loaded context is smaller than this, the server clamps it.
MAX_COMPLETION_TOKENS = 8000


def call_llm(prompt, model=None):
    model = model or MODEL
    resp = _client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
        max_completion_tokens=MAX_COMPLETION_TOKENS,
    )
    choice = resp.choices[0]
    content = (choice.message.content or "").strip()
    if not content:
        reasoning = getattr(choice.message, "reasoning_content", "") or ""
        usage = getattr(resp, "usage", None)
        usage_summary = f"{usage}" if usage else "<no usage>"
        raise RuntimeError(
            f"Model returned empty content (finish_reason={choice.finish_reason!r}). "
            f"Usage: {usage_summary}. "
            f"Reasoning length: {len(reasoning)} chars. "
            f"For reasoning models, raise LM Studio's loaded context window "
            f"(qwen3.6-27b supports ~18k) or lower the upstream chain size."
        )
    return content


def write_stage(slug, folder, prompt_path, content):
    if not content.strip():
        raise ValueError("refusing to write an empty stage artifact")
    out = SPECS_ROOT / folder / f"{slug}.md"
    meta = SPECS_ROOT / folder / f"{slug}.meta.yaml"
    if out.exists() and not CONFIRM_OVERWRITE:
        raise FileExistsError(
            f"{out} exists; set CONFIRM_OVERWRITE = True in cell 1 to overwrite."
        )
    out.write_text(content)
    meta_data = {
        "model": MODEL,
        "provider": "lmstudio",
        "endpoint": LMSTUDIO_HOST,
        "prompt_path": str(prompt_path.relative_to(WORKSPACES_ROOT)),
        "prompt_sha256": hashlib.sha256(prompt_path.read_bytes()).hexdigest(),
        "promoted_at": _dt.datetime.now(_dt.UTC).replace(microsecond=0).isoformat().replace("+00:00", "Z"),
    }
    meta.write_text(yaml.safe_dump(meta_data, sort_keys=False))
    print(f"wrote {out.relative_to(WORKSPACES_ROOT)}")
    print(f"wrote {meta.relative_to(WORKSPACES_ROOT)}")


def stage_from_index(folder):
    return [i for i, (f, _, _) in enumerate(STAGES) if f == folder][0]


_START_INDEX = stage_from_index(START_STAGE)
print(f"start at stage {_START_INDEX}: {START_STAGE}")

## Stage 02 — Research

Surveys repo assets relevant to the idea. Reads only `01_ideas/<slug>.md`.

In [5]:
# Preview
_folder, _prompt_path, _var = STAGES[1]
_chain = read_chain(SLUG)
_rendered = render_prompt(_prompt_path, _chain)
print(_rendered)

# Spec promotion — Research stage

You are a spec promotion assistant. You have been given an idea from a
local-first AI orchestration repository. Survey what already exists in the
repository that bears on the idea, and surface the questions that the
requirements stage must answer.

Do not make decisions in this stage. Decisions belong to requirements. This
stage only reports.

Produce a Markdown document that:

1. Starts with the heading `# Research — <slug>` where `<slug>` is implied
   by the idea title (kebab-case).
2. Has a section listing existing repo assets the idea must reuse or respect.
   Reference files by relative path when you can infer them from the idea.
3. Has a section listing layer-ownership constraints if the idea touches
   buckets like `02_core`, `03_adapters`, `04_harnesses`, `05_router`,
   `06_workflows`, `08_drivers`, `11_mcp`, or `12_prompts`.
4. Has a section listing external dependencies (services, models, libraries).
5. Ends with a numbered list of open qu

In [6]:
# Run
if _START_INDEX <= 1:
    _content = call_llm(_rendered)
    print(_content[:1500])
    write_stage(SLUG, _folder, _prompt_path, _content)
else:
    print(f"skipped (START_STAGE = {START_STAGE})")



# Research — missing-idea

## Existing Repo Assets
The idea content was not provided. The input ends with the placeholder `{{idea}}`. No assets can be surveyed without the idea description.

## Layer-Ownership Constraints
No idea provided. Layer constraints cannot be determined.

## External Dependencies
No idea provided. External dependencies cannot be identified.

## Open Questions
1. Please provide the idea text to proceed with the research stage.
wrote 00_specs/02_research/create-format-markdown-workflow.md.md
wrote 00_specs/02_research/create-format-markdown-workflow.md.meta.yaml


/var/folders/8b/3fq8bjr561q96rdngss7c4x40000gp/T/ipykernel_15494/1770065628.py:49: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "promoted_at": _dt.datetime.utcnow().replace(microsecond=0).isoformat() + "Z",


## Stage 03 — Requirements

In [ ]:
_folder, _prompt_path, _var = STAGES[2]
_chain = read_chain(SLUG)
_rendered = render_prompt(_prompt_path, _chain)
print(_rendered)

In [ ]:
if _START_INDEX <= 2:
    _content = call_llm(_rendered)
    print(_content[:1500])
    write_stage(SLUG, _folder, _prompt_path, _content)
else:
    print(f"skipped (START_STAGE = {START_STAGE})")

## Stage 04 — Tasks

In [ ]:
_folder, _prompt_path, _var = STAGES[3]
_chain = read_chain(SLUG)
_rendered = render_prompt(_prompt_path, _chain)
print(_rendered)

In [ ]:
if _START_INDEX <= 3:
    _content = call_llm(_rendered)
    print(_content[:1500])
    write_stage(SLUG, _folder, _prompt_path, _content)
else:
    print(f"skipped (START_STAGE = {START_STAGE})")

## Stage 05 — Prompts

In [ ]:
_folder, _prompt_path, _var = STAGES[4]
_chain = read_chain(SLUG)
_rendered = render_prompt(_prompt_path, _chain)
print(_rendered)

In [ ]:
if _START_INDEX <= 4:
    _content = call_llm(_rendered)
    print(_content[:1500])
    write_stage(SLUG, _folder, _prompt_path, _content)
else:
    print(f"skipped (START_STAGE = {START_STAGE})")

## Stage 06 — Final

Bundles the chain into a frozen handoff prompt plus a stage index.

In [ ]:
_folder, _prompt_path, _var = STAGES[5]
_chain = read_chain(SLUG)
_rendered = render_prompt(_prompt_path, _chain)
print(_rendered)

In [ ]:
if _START_INDEX <= 5:
    _content = call_llm(_rendered)
    print(_content[:1500])
    write_stage(SLUG, _folder, _prompt_path, _content)
else:
    print(f"skipped (START_STAGE = {START_STAGE})")

## Verify the promotion landed

Acceptance A2: every stage from 01 to 06 has a `<slug>.md`, and every stage from 02 onward has a matching `.meta.yaml`.

In [ ]:
import subprocess

result = subprocess.run(
    ["find", str(SPECS_ROOT), "-name", f"{SLUG}*"],
    check=True,
    capture_output=True,
    text=True,
)
for line in sorted(result.stdout.splitlines()):
    print(line.replace(str(WORKSPACES_ROOT) + "/", ""))

In [ ]:
# Show the frozen handoff so you can copy it into a fresh agent session.
print((SPECS_ROOT / "06_final" / f"{SLUG}.md").read_text())